# Suite1 Project2 Preprocessing

**Dataset:** California Housing

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 2: Data Preprocessing Pipeline"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('housing.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite1_project2_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)  # for reproducibility of missing value injection

## PROJECT 2: DATA PREPROCESSING PIPELINE

In [ ]:
# ── 1. Introduce missing values for demonstration ──

## 1. Data Quality Issues — Introducing Realistic Problems

In [ ]:
df_dirty = df.copy()
# Inject missing values randomly (5% in selected columns)
for col in ['AveRooms', 'AveBedrms']:
    mask = np.random.random(len(df_dirty)) < 0.05
    df_dirty.loc[mask, col] = np.nan

# Inject outliers in AveRooms
outlier_idx = np.random.choice(len(df_dirty), 20, replace=False)
df_dirty.loc[outlier_idx, 'AveRooms'] = df_dirty['AveRooms'].max() * np.random.uniform(3, 8, 20)

# Inject an impossible value
df_dirty.loc[np.random.randint(0, len(df_dirty)), 'AveBedrms'] = -1

print("=== Data Quality Report ===")
print(f"Missing values:")
print(df_dirty.isnull().sum().to_string())
print(f"\nTotal missing: {df_dirty.isnull().sum().sum():,} / {df_dirty.size:,}")
missing_pct = df_dirty.isnull().sum().sum() / df_dirty.size * 100
print(f"Missing percentage: {missing_pct:.2f}%")

# ── 2. Missing Value Handling ──

## 2. Handling Missing Values

In [ ]:
# Option A: Drop rows with missing (if few)
df_drop = df_dirty.dropna()
print(f"After dropna(): {len(df_dirty)} → {len(df_drop)} rows (lost {len(df_dirty)-len(df_drop)})")

# Option B: Fill with mean/median
df_filled = df_dirty.copy()
for col in ['AveRooms', 'AveBedrms']:
    median_val = df_filled[col].median()
    df_filled[col] = df_filled[col].fillna(median_val)
    print(f"Filled '{col}' missing with median = {median_val:.3f}")

print(f"\nAfter filling: {df_filled.isnull().sum().sum()} remaining missing values")

# ── 3. Outlier Detection ──

## 3. Outlier Detection — IQR Method

In [ ]:
num_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'MedHouseVal']
outlier_report = []
for col in num_cols:
    Q1 = df_filled[col].quantile(0.25)
    Q3 = df_filled[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_filled[col] < lower) | (df_filled[col] > upper)).sum()
    pct = n_outliers / len(df_filled) * 100
    outlier_report.append({'Column': col, 'Q1': round(float(Q1),3), 'Q3': round(float(Q3),3), 
                          'IQR': round(float(IQR),3), 'Lower Fence': round(float(lower),3), 
                          'Upper Fence': round(float(upper),3), 'Outliers': int(n_outliers), '%': f'{pct:.1f}%'})
    print(f"  {col:15s}: Q1={Q1:8.3f} Q3={Q3:8.3f} IQR={IQR:8.3f} | Outliers={n_outliers:5d} ({pct:.1f}%)")

outlier_df = pd.DataFrame(outlier_report)
    json.dump(outlier_report, f, indent=2)

# Visualize outliers
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(y=df_filled[col], ax=axes[i], color='#2e86de')
    axes[i].set_title(col, fontsize=11)
if len(num_cols) < len(axes):
    axes[-1].axis('off')
plt.suptitle('Outlier Detection — Box Plots (Numerical Features)', fontsize=14)
plt.tight_layout()
# 'p2_outlier_boxplots')
plt.show()
print("[Saved] p2_outlier_boxplots.png")

# ── 4. Feature Selection ──

## 4. Feature Selection Strategies

In [ ]:
X = df_filled[num_cols].copy()
y = X.pop('MedHouseVal')

# 4a. Filter Method: Correlation-based
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
print("\n=== 4a. Filter Method — Correlation with Target ===")
print(corr_with_target.to_string())

# 4b. SelectKBest
selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X, y)
kbest_scores = pd.DataFrame({'Feature': X.columns, 'F_Score': selector.scores_, 'P_Value': selector.pvalues_})
kbest_scores = kbest_scores.sort_values('F_Score', ascending=False)
print(f"\n=== 4b. SelectKBest (F-Regression) ===")
print(kbest_scores.to_string(index=False))

# 4c. RFE (Wrapper method)
model = LinearRegression()
rfe = RFE(estimator=model, n_features_to_select=4)
rfe.fit(X, y)
rfe_ranking = pd.DataFrame({'Feature': X.columns, 'Selected': rfe.support_, 'Rank': rfe.ranking_})
print(f"\n=== 4c. RFE — Recursive Feature Elimination ===")
print(rfe_ranking.to_string(index=False))

# ── 5. Feature Scaling ──

## 5. Scaling — Normalization & Standardization

In [ ]:
X_sample = X.head(10)

scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X_sample)

scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X_sample)

print("Original (first 5 rows, 3 features):")
print(X_sample[['MedInc', 'HouseAge', 'Population']].head().to_string())
print("\nAfter StandardScaler (mean=0, std=1):")
std_df = pd.DataFrame(X_std[:5, :3], columns=['MedInc', 'HouseAge', 'Population'])
print(std_df.round(3).to_string())
print(f"\n  Mean ≈ {X_std[:5, :3].mean():.6f}, Std ≈ {X_std[:5, :3].std(ddof=0):.6f}")
print("\nAfter MinMaxScaler (range [0,1]):")
mm_df = pd.DataFrame(X_mm[:5, :3], columns=['MedInc', 'HouseAge', 'Population'])
print(mm_df.round(3).to_string())

# Visualize scaling
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(axes, [X_sample.values, X_std, X_mm], 
                           ['Original', 'Standardized (Z-score)', 'Normalized [0,1]']):
    sns.boxplot(data=pd.DataFrame(data, columns=X.columns), ax=ax, palette='Set2')
    ax.set_title(title, fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
# 'p2_scaling_comparison')
plt.show()
print("[Saved] p2_scaling_comparison.png")

# ── 6. Dimensionality Reduction (PCA) ──

## 6. Dimensionality Reduction — PCA

In [ ]:
# Scale full data first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
pca_data = pca.fit_transform(X_scaled)
pca_components_count = len(pca.explained_variance_ratio_)

print(f"Original dimensions: {X.shape[1]}")
print(f"Components with 95% variance: {(pca.explained_variance_ratio_.cumsum() < 0.95).sum() + 1}")
print(f"\nExplained variance ratio per component:")
for i, (ev, cev) in enumerate(zip(pca.explained_variance_ratio_, np.cumsum(pca.explained_variance_ratio_))):
    print(f"  PC{i+1}: {ev:.4f} ({cev:.4f} cumulative)")

# PCA Scree + Cumulative
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, pca_components_count+1), pca.explained_variance_ratio_, color='#2e86de', alpha=0.7)
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=13)
axes[0].axhline(y=0.05, color='red', linestyle='--', alpha=0.5)

axes[1].plot(range(1, pca_components_count+1), np.cumsum(pca.explained_variance_ratio_), 
             'o-', color='#e74c3c', linewidth=2)
axes[1].axhline(y=0.95, color='green', linestyle='--', alpha=0.7, label='95% threshold')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Explained Variance', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
# 'p2_pca_analysis')
plt.show()
print("[Saved] p2_pca_analysis.png")

# PCA 2D visualization
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(pca_data[:, 0], pca_data[:, 1], c=y, cmap='viridis', alpha=0.4, s=5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('PCA 2D Projection — Colored by House Value', fontsize=14)
plt.colorbar(sc, ax=ax, label='MedHouseVal ($100K)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p2_pca_2d')
plt.show()
print("[Saved] p2_pca_2d.png")

# ── Export Results ──

## 7. Exporting Results

In [ ]:
results = {
    'missing_report': df_dirty.isnull().sum().to_dict(),
    'outlier_report': outlier_report,
    'feature_selection': {
        'correlation': corr_with_target.to_dict(),
        'rfe_selected': rfe.support_.tolist(),
    },
    'pca': {
        'explained_variance_ratio': pca.explained_variance_ratio_.tolist(),
        'cumulative': np.cumsum(pca.explained_variance_ratio_).tolist(),
        'n_features': X.shape[1],
        'n_components_95pct': int((pca.explained_variance_ratio_.cumsum() < 0.95).sum() + 1)
    }
}

    json.dump(results, f, indent=2, default=str)

print("Done.")